# Try an Open Data Analysis yourself!

## The Higgs to four lepton analysis from the ATLAS Open Data release of 2020, with RDataFrame


This exercise is the Higgs to four lepton analysis from the ATLAS Open Data release in 2020 (http://opendata.atlas.cern/release/2020/documentation/). The data was taken with the ATLAS detector during 2016 at a center-of-mass energy of 13 TeV. The decay of the Standard Model Higgs boson to two Z bosons and subsequently to four leptons is called the "golden channel". The selection leads to a narrow invariant mass peak on top a relatively smooth and small background, revealing the Higgs at 125 GeV. Systematic errors for the MC scale factors are computed and the Vary function of RDataFrame is used for plotting. The analysis is translated to an RDataFrame workflow processing about 300 MB of simulated events and data.

There are quite a few steps in this analysis. We have covered the concepts during the course, we will guide you so that you can write your own code where needed. Can you get to the final plot with the Higgs boson peak? 

In case you get stuck, you can find the solution in the same directory.


In [ ]:
import ROOT
import os

The analysis needs custom code written in a separate header, so we need to first include it here

In [ ]:
# The header file that needs to be included is 'utils.h'

# {PLACEHOLDER for your code}

## Specifying the input dataset

In this exercise, we use a JSON config file to define the dataset specification. It contains the names of the files, together with metadata associated with different samples. The metadata contained in the JSON file is accessible within a [`DefinePerSample`](https://root.cern/doc/v628/classROOT_1_1RDF_1_1RInterface.html#a29d77593e95c0f84e359a802e6836a0e) call, through the [`RSampleInfo`](https://root.cern/doc/master/classROOT_1_1RDF_1_1RSampleInfo.html) class.

In [ ]:
# The necessary JSON file is: "../data/dataset_spec.json". You can explore what it looks like. 
# Focus on the metadata information? You should find 4 metadata instances per sample. 
# Create an RDataFrame first, then, use DefinePerSample to define the columns 
# for each of the metadata instances, for example, the cross section. 

# {PLACEHOLDER for your code}

In [ ]:
# We also define a scaling factor using one of the functions from utils.h, it is also different for 
# Data and MC that's why we use DefinePerSample method as well. 

df = df.DefinePerSample("scale", "scale(rdfslot_, rdfsampleinfo_)")

## Perform selections

We need to select events with exactly four good leptons conserving charge and lepton numbers. Note that all collections read by RDataFrame are implicitly handled as the `ROOT::RVec` type. Let's go step by step.

In [ ]:
# Firstly, select electron OR muon trigger

# {PLACEHOLDER for your code}

In [ ]:
# Secondly, we want to select "good leptons" within the events
# (hint: operations on collections)
# We want to define a "good_lep" mask.
# We define a good lepton as a lepton which has:
# absolute eta below 2.5, 
# pt greater than 5000
# and obeys both of the isolation criteria:
# (pt cone of 30 / pt) must be less than 0.3 and
# (pt cone of 20 / pt) must be less than 0.3. 

# {PLACEHOLDER for your code}

In [ ]:
# Once you define the masking, it is time to select events of interest.
# We want events in which we have 4 good leptons whose charges cancel out.
# (hint - 2 Filter expressions are needed). 

# {PLACEHOLDER for your code}

In [ ]:
# Finally, use the fact that the PDG numbers of the lepton types are set to 
# 11 or 13 for an electron or muon irrespective of the charge. 
# Remember that you have 4 leptons in total and pairs 
# of leptons must be of the same type, e.g. 2 electron-positron pairs, 
# 2 muon-antimuon pairs or one electron-positron pair and one muon-antimuon pair.
# Hint - first, define the column which calculated the total PDG number of all leptons in the event,
# secondly, construct the filter. 

# {PLACEHOLDER for your code}

In [ ]:
# Now we apply additional cuts depending on lepton flavour.
# Given the function GoodElectronsAndMuons defined in utils.h, 
# complete the Filter expression below. 
df = df.Filter(
    "GoodElectronsAndMuons( PLACEHOLDER )"
)

# Now we can create new columns with the kinematics of good leptons
df = (
    df.Define("goodlep_pt", "lep_pt[good_lep]")
    .Define("goodlep_eta", "lep_eta[good_lep]")
    .Define("goodlep_phi", "lep_phi[good_lep]")
    .Define("goodlep_E", "lep_E[good_lep]")
    .Define("goodlep_type", "lep_type[good_lep]")
)

In [ ]:
# We want to select events in which the first 3 leptons have high transverse momenta (goodlep_pt)
# The first one is above 25000, second above 15000 and third above 10000

# {PLACEHOLDER for your code}

## Define final observables, prepare histograms, compute variations in MC event weights.

In [ ]:
# Reweighting of the samples is different for "data" and "MC".
# We use DefinePerSample to define which samples are MC and hence need reweighting
df = df.DefinePerSample("isMC", 'rdfsampleinfo_.Contains("mc")')

# Now we will create a weight column, using "weights" function from utils.h which will compute the scale factor for MC.
# For data, no scaling is needed. 
df = df.Define(
    "weight",
    "double x; return isMC ? weights(scaleFactor_ELE, scaleFactor_MUON, scaleFactor_LepTRIGGER, scaleFactor_PILEUP, scale, mcWeight, xsecs, sumws, lumi) :  1.;",
)

In [ ]:

# Finally, we need to define the invariant mass of the four leptons. 
# Again, we can use the function defined in the utils.h

# {PLACEHOLDER for your code}



In [ ]:
# We can now book histograms for the four different samples: data, higgs, 
# zz and other (this is specific to this particular analysis)
histos = []
for sample_category in ["data", "higgs", "zz", "other"]:
    histos.append(
        df.Filter(f'sample_category == "{sample_category}"').Histo1D(
            ROOT.RDF.TH1DModel(f"{sample_category}", "m4l", 24, 80, 170),
            "m4l",
            "weight",
        )
    )

In [ ]:
# We will now create one helper object to run the interpolation for the systematic uncertainty evaluation in the event loop.
# Again, to see the details of the class VaryHelper, refer to the utils.h 
ROOT.gInterpreter.ProcessLine("VaryHelper variationsFactory;")

In [ ]:
# Use the Vary method to add the systematic variations to the total MC scale factor ("weight") of the analysis.
# Note, variations are applicable to the MC events only. 

df_variations_mc = (
    df.Filter("isMC == true")
    .Vary( #{PLACEHOLDER for your code} 
    )
    .Histo1D(ROOT.RDF.TH1DModel("Invariant Mass", "m4l", 24, 80, 170), "m4l", "weight")
)

# We want to end up with varied histograms which we can use for making the final plot. 
histos_mc = ROOT.RDF.Experimental.VariationsFor(df_variations_mc)

## Retrieving histograms and plotting

We reached the end of the analysis part. We now evaluate the total MC uncertainty based on the variations. No computation graph was triggered yet, we trigger the computation graph for all histograms at once now, by calling 'histos_mc["nominal"].GetXaxis()'. Note, in this case the uncertainties are symmetric.

In [ ]:
# Use the varied histograms to compute the bin error

for i in range(0, histos_mc["nominal"].GetXaxis().GetNbins()):
    (
        histos_mc["nominal"].SetBinError(
            i, (histos_mc["weight:up"].GetBinContent(i) - histos_mc["nominal"].GetBinContent(i))
        )
    )

Make the plot of the data, individual MC contributions and the total MC scale factor systematic variations.

In [ ]:
ROOT.gROOT.SetStyle("ATLAS")

# Create canvas with pad
c4 = ROOT.TCanvas("c4", "", 620, 620)
pad = ROOT.TPad("upper_pad", "", 0, 0, 1, 1)
pad.SetTickx(False)
pad.SetTicky(False)
pad.Draw()
pad.cd()

# Draw stack with MC contributions
stack = ROOT.THStack()

# Retrieve values of the data and MC histograms in order to plot them.
# Draw cloned histograms to preserve graphics when original object goes out of scope
# Note: GetValue() action operation is performed after all lazy actions of the RDF were defined first.

h_data = histos[0].GetValue().Clone()
h_higgs = histos[1].GetValue().Clone()
h_zz = histos[2].GetValue().Clone()
h_other = histos[3].GetValue().Clone()

for h, color in zip([h_other, h_zz, h_higgs], [ROOT.kViolet-9, ROOT.kAzure-9, ROOT.kRed+2]):
    h.SetLineWidth(1)
    h.SetLineColor(1)
    h.SetFillColor(color)
    stack.Add(h)

stack.Draw("HIST")
stack.GetXaxis().SetLabelSize(0.04)
stack.GetXaxis().SetTitleSize(0.045)
stack.GetXaxis().SetTitleOffset(1.3)
stack.GetXaxis().SetTitle("m_{4l}^{H#rightarrow ZZ} [GeV]")
stack.GetYaxis().SetLabelSize(0.04)
stack.GetYaxis().SetTitleSize(0.045)
stack.GetYaxis().SetTitle("Events")
stack.SetMaximum(35)
stack.GetYaxis().ChangeLabel(1, -1, 0)

# Draw MC scale factor and variations
histos_mc["nominal"].SetFillColor(ROOT.kBlack)
histos_mc["nominal"].SetFillStyle(3254)
h_nominal = histos_mc["nominal"].DrawClone("E2 same")
histos_mc["weight:up"].SetLineColor(ROOT.kGreen+2)
h_weight_up = histos_mc["weight:up"].DrawClone("HIST SAME")
histos_mc["weight:down"].SetLineColor(ROOT.kBlue+2)
h_weight_down = histos_mc["weight:down"].DrawClone("HIST SAME")

# Draw data histogram
h_data.SetMarkerStyle(20)
h_data.SetMarkerSize(1.2)
h_data.SetLineWidth(2)
h_data.SetLineColor(ROOT.kBlack)
h_data.Draw("E SAME")  # Draw raw data with errorbars

# Add legend
legend = ROOT.TLegend(0.57, 0.65, 0.94, 0.94)
legend.SetTextFont(42)
legend.SetFillStyle(0)
legend.SetBorderSize(0)
legend.SetTextSize(0.02)
legend.SetTextAlign(32)
legend.AddEntry(h_data, "Data", "lep")
legend.AddEntry(h_higgs, "Higgs MC", "f")
legend.AddEntry(h_zz, "ZZ MC", "f")
legend.AddEntry(h_other, "Other MC", "f")
legend.AddEntry(h_weight_down, "Total MC Variations Down", "l")
legend.AddEntry(h_weight_up, "Total MC Variations Up", "l")
legend.AddEntry(h_nominal, "Total MC Uncertainty", "f")
legend.Draw()

text = ROOT.TLatex()
text.SetTextFont(72)
text.SetTextSize(0.04)
text.DrawLatexNDC(0.19, 0.85, "ATLAS")
text.SetTextFont(42)
text.DrawLatexNDC(0.19 + 0.15, 0.85, "Open Data")
text.SetTextSize(0.035)
text.DrawLatexNDC(0.21, 0.80, "#sqrt{s} = 13 TeV, 10 fb^{-1}")

c4.Draw()